In [1]:
import os 
import random
import pandas as pd
random.seed(33)  # For reproducibility

In [2]:
annotators = ["A1", "A2", "A3", "A4", "A5"]
newspapers = ["corriere_della_sera", "repubblica", "il_gazzettino", "ilmessaggero", "lastampa"]
NUM_VIDEOS_PER_NEWSPAPER = 6

In [3]:
newspaper_selected_videos = {}

for newspaper in newspapers:
    input_dir = f"VideosComments/youtube/comments_anonymized/{newspaper}"
    comments_files = [f for f in os.listdir(input_dir) if f.endswith(".csv")]

    video_comments_count = {}
    for comments_file in comments_files:
        video_id = comments_file.split(".csv")[0]
        df = pd.read_csv(os.path.join(input_dir, comments_file))
        video_comments_count[video_id] = len(df)

    selected_videos = {
        video_id: count
        for video_id, count in video_comments_count.items()
        if 20 <= count <= 50
    }

    if len(selected_videos) < NUM_VIDEOS_PER_NEWSPAPER:
        print(f"Warning: Not enough videos for {newspaper}. Found {len(selected_videos)} videos.")
        # Add those restricted to 15 comments
        for video_id, count in video_comments_count.items():
            if 15 <= count < 20:
                selected_videos[video_id] = count
        if len(selected_videos) < NUM_VIDEOS_PER_NEWSPAPER:
            print(f"Still not enough videos for {newspaper}. Found {len(selected_videos)} videos after adding 15-19 comments.")
            # Add those restricted to 10 comments
            for video_id, count in video_comments_count.items():
                if 10 <= count < 15:
                    selected_videos[video_id] = count
    
    n_to_sample = min(NUM_VIDEOS_PER_NEWSPAPER, len(selected_videos))
    selected_video_ids = random.sample(list(selected_videos.keys()), n_to_sample)
    newspaper_selected_videos[newspaper] = selected_video_ids


Still not enough videos for lastampa. Found 5 videos after adding 15-19 comments.


In [4]:
newspaper_selected_videos

{'corriere_della_sera': ['mPhV_I9T4vc',
  'JnLp9NYdT1E',
  '-Jz16gcNDG4',
  'UzDNbU-_zl4',
  'lSaNEVCmKmY',
  'NrEDu1agMz4'],
 'repubblica': ['7FfdwuricJk',
  'y-_N-NGhP7k',
  '4cvfxPJACdc',
  'MHcD77-9TpY',
  'tJEj5q5Jzgs',
  'tJGcw5TVtvY'],
 'il_gazzettino': ['ziamKdMMXTA',
  'dInka8SPF6Y',
  '5s85hZ-6qzw',
  'co7cT93lKDQ',
  'NrsOQsfaYYw',
  'VCwqO9o8mLc'],
 'ilmessaggero': ['7llrt4puUhA',
  'yhQY_bLvUW4',
  'ghcxpoGbxTA',
  '0paspwjsZmE',
  'Re8gVvm24ZU',
  'K6XtSG4sox4'],
 'lastampa': ['y1Qb9O7YhHM',
  'nkY13PRpl4w',
  'miSx-D39osw',
  'dXVH1T_3XGA',
  'm4LD6miuPVM',
  'dydwzHnuhxE']}

In [5]:
import random
from itertools import combinations

annotator_triplets = list(combinations(annotators, 3))

# flatten all videos across newspapers, keeping (newspaper, id)
all_videos = []
for newspaper in newspapers:
    all_videos.extend(
        (newspaper, video_id)
        for video_id in newspaper_selected_videos[newspaper]
    )

# sanity check
assert len(all_videos) == 30, f"Expected 30 videos, got {len(all_videos)}"

# shuffle for randomness
random.shuffle(all_videos)

# assign 3 per triplet (now each item is (newspaper, id))
triplet_video_assignments = {
    triplet: all_videos[i * 3:(i + 1) * 3]
    for i, triplet in enumerate(annotator_triplets)
}

In [6]:
triplet_video_assignments

{('A1', 'A2', 'A3'): [('ilmessaggero', 'ghcxpoGbxTA'),
  ('il_gazzettino', 'NrsOQsfaYYw'),
  ('il_gazzettino', 'ziamKdMMXTA')],
 ('A1', 'A2', 'A4'): [('ilmessaggero', 'Re8gVvm24ZU'),
  ('il_gazzettino', 'dInka8SPF6Y'),
  ('lastampa', 'm4LD6miuPVM')],
 ('A1', 'A2', 'A5'): [('ilmessaggero', '7llrt4puUhA'),
  ('lastampa', 'dXVH1T_3XGA'),
  ('ilmessaggero', 'yhQY_bLvUW4')],
 ('A1', 'A3', 'A4'): [('corriere_della_sera', 'lSaNEVCmKmY'),
  ('lastampa', 'dydwzHnuhxE'),
  ('il_gazzettino', 'co7cT93lKDQ')],
 ('A1', 'A3', 'A5'): [('il_gazzettino', '5s85hZ-6qzw'),
  ('corriere_della_sera', 'mPhV_I9T4vc'),
  ('corriere_della_sera', 'NrEDu1agMz4')],
 ('A1', 'A4', 'A5'): [('repubblica', '4cvfxPJACdc'),
  ('lastampa', 'miSx-D39osw'),
  ('ilmessaggero', 'K6XtSG4sox4')],
 ('A2', 'A3', 'A4'): [('lastampa', 'nkY13PRpl4w'),
  ('repubblica', 'tJGcw5TVtvY'),
  ('corriere_della_sera', '-Jz16gcNDG4')],
 ('A2', 'A3', 'A5'): [('repubblica', '7FfdwuricJk'),
  ('repubblica', 'MHcD77-9TpY'),
  ('repubblica', 'tJEj5

In [7]:
# Now for each annotator, give the assigned videos 
annotator_video_assignments = {annotator: [] for annotator in annotators}

for triplet, videos in triplet_video_assignments.items():
    for annotator in triplet:
        annotator_video_assignments[annotator].extend(
            (newspaper, video_id) for (newspaper, video_id) in videos
        )

In [8]:
import os
import shutil

source_base = "VideosComments/youtube"
output_root = "Annotators_workload"

for annotator, items in annotator_video_assignments.items():

    for newspaper, video_id in items:

        # destination paths (FIXED ROOT)
        dest_csv_dir = os.path.join(
            output_root,
            annotator,
            "VideosComments", "youtube", "comments_anonymized", newspaper
        )

        dest_json_dir = os.path.join(
            output_root,
            annotator,
            "VideosComments", "youtube", "metadata", newspaper
        )

        os.makedirs(dest_csv_dir, exist_ok=True)
        os.makedirs(dest_json_dir, exist_ok=True)

        # source files
        src_csv = os.path.join(
            source_base,
            "comments_anonymized",
            newspaper,
            f"{video_id}.csv"
        )

        src_json = os.path.join(
            source_base,
            "metadata",
            newspaper,
            f"{video_id}.json"
        )

        # destination files
        dest_csv = os.path.join(dest_csv_dir, f"{video_id}.csv")
        dest_json = os.path.join(dest_json_dir, f"{video_id}.json")

        # copy safely
        if os.path.exists(src_csv):
            shutil.copy2(src_csv, dest_csv)
        else:
            print(f"[WARN] Missing CSV: {src_csv}")

        if os.path.exists(src_json):
            shutil.copy2(src_json, dest_json)
        else:
            print(f"[WARN] Missing JSON: {src_json}")

    # -----------------------------
    # EMPTY annotated folder (FIXED ROOT)
    # -----------------------------
    os.makedirs(
        os.path.join(
            output_root,
            annotator,
            "VideosComments", "youtube", "annotated_comments"
        ),
        exist_ok=True
    )

print("All annotator folders created under Annotators_workload.")

All annotator folders created under Annotators_workload.


### Create pilot

In [28]:
# Select 2 videos total for pilot annotation
pilot_videos = []

all_selected_videos = []

for newspaper in newspapers:
    input_dir = f"VideosComments/youtube/comments_anonymized/{newspaper}"
    comments_files = [f for f in os.listdir(input_dir) if f.endswith(".csv")]

    video_comments_count = {}

    for comments_file in comments_files:
        video_id = comments_file.split(".csv")[0]
        df = pd.read_csv(os.path.join(input_dir, comments_file))
        video_comments_count[video_id] = len(df)

    selected_videos = [
        (newspaper, video_id)
        for video_id, count in video_comments_count.items()
        if 20 <= count <= 30
        and video_id not in newspaper_selected_videos[newspaper]
    ]

    all_selected_videos.extend(selected_videos)

if len(all_selected_videos) < 2:
    print(f"Warning: Only found {len(all_selected_videos)} eligible videos.")

pilot_videos = random.sample(
    all_selected_videos,
    min(2, len(all_selected_videos))
)

print(pilot_videos)

[('corriere_della_sera', 'CAVns3bw4tc'), ('repubblica', 'CClbcflccg8')]


In [29]:
# Place pilot videos in Annotators_workload/PILOT folder
pilot_output_root = os.path.join(output_root, "PILOT")
os.makedirs(pilot_output_root, exist_ok=True)

for newspaper, video_id in pilot_videos:

    # destination paths (FIXED ROOT)
    dest_csv_dir = os.path.join(
        pilot_output_root,
        "VideosComments", "youtube", "comments_anonymized", newspaper
    )

    dest_json_dir = os.path.join(
        pilot_output_root,
        "VideosComments", "youtube", "metadata", newspaper
    )

    os.makedirs(dest_csv_dir, exist_ok=True)
    os.makedirs(dest_json_dir, exist_ok=True)

    # source files
    src_csv = os.path.join(
        source_base,
        "comments_anonymized",
        newspaper,
        f"{video_id}.csv"
    )

    src_json = os.path.join(
        source_base,
        "metadata",
        newspaper,
        f"{video_id}.json"
    )

    # destination files
    dest_csv = os.path.join(dest_csv_dir, f"{video_id}.csv")
    dest_json = os.path.join(dest_json_dir, f"{video_id}.json")

    # copy safely
    if os.path.exists(src_csv):
        shutil.copy2(src_csv, dest_csv)
    else:
        print(f"[WARN] Missing CSV: {src_csv}")

    if os.path.exists(src_json):
        shutil.copy2(src_json, dest_json)
    else:
        print(f"[WARN] Missing JSON: {src_json}")


In [24]:
pilot_videos

[('corriere_della_sera', 'EYMrDAN1H4U'), ('ilmessaggero', 'Rm-8FjCgI0k')]